## Tasks 1 and 2(Ingestion and Cleaning)
Task 01: Write explicit StructType schemas for all 4 CSVs. Load them with header=True but no inferSchema. Any row that fails casting goes into a "rejected Dataframe which you log separately.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
spark = SparkSession.builder.getOrCreate()

In [2]:
customerStruct = StructType([
    StructField('customer_id', StringType(), True),
    StructField('customer_name', StringType(), True),
    StructField('email', StringType(), True),
    StructField('country', StringType(), True),
    StructField('customer_tier', StringType(), True),
    StructField('signup_date', StringType(), True),
])

In [3]:
df_customers = spark.read.format('csv')\
                .schema(customerStruct)\
                .option('inferschema', False)\
                .option('header', True)\
                .load('C:/Users/ADMIN/Desktop/Lux Dev Data Enginering/Assignment_1/data/customers.csv')

In [ ]:
val = [df_customers.collect()[0]["signup_date"]]
print(val)
formatted_date = []

# Check if the date value is not None before processing
if val[0] is not None and '/' in val[0]:
    year = val[0][6:10]
    month = val[0][3:5]
    day = val[0][0:2]
    new_date = f'{year}-{month}-{day}'
    print(new_date)
    formatted_date.append(new_date)
else:
    # Handle the case when date is None or doesn't contain '/'
    if val[0] is not None:
        formatted_date.append(val[0])  # Keep original format if no '/'
    else:
        formatted_date.append("No date available")  # Handle None case

print(formatted_date)

In [ ]:
df_customers.show()

In [4]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_tier: string (nullable = true)
 |-- signup_date: string (nullable = true)



In [5]:
orderItemStruct = StructType([
    StructField('item_id', StringType(), True),
    StructField('order_id', StringType(), True),
    StructField('product_name', StringType(), True),
    StructField('category', StringType(), True),
    StructField('quantity', IntegerType(), True),
    StructField('unit_price', FloatType(), True),
])

In [6]:
df_order_items = spark.read.format('csv')\
                .schema(orderItemStruct)\
                .option('inferschema', False)\
                .option('header', True)\
                .load('C:/Users/ADMIN/Desktop/Lux Dev Data Enginering/Assignment_1/data/order_items.csv')

In [ ]:
df_order_items.show()

In [ ]:
df_order_items.printSchema()

In [7]:
orderStruct = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("status", StringType(), True),
    StructField("total_amount", FloatType(), True),
    StructField("discount_pct",IntegerType(), True)
    ])

In [8]:
df_orders = spark.read.format('csv')\
                .schema(orderStruct)\
                .option('inferschema', False)\
                .option('header', True)\
                .load('C:/Users/ADMIN/Desktop/Lux Dev Data Enginering/Assignment_1/data/orders.csv')
df_orders.show()

+--------+-----------+----------+---------+------------+------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|
+--------+-----------+----------+---------+------------+------------+
| O000001|     C00078|03/08/2022|cancelled|      230.07|           5|
| O000002|     C00044|22/01/2021|completed|      221.77|           5|
| O000003|     C00286|2024-10-17|  pending|      1795.1|          15|
| O000004|     C00398|16/05/2022| refunded|      404.43|          20|
| O000005|     C00231|21/03/2024|cancelled|      -26.78|          15|
| O000006|     C00302|18/07/2024|cancelled|      123.89|          15|
| O000007|     C00292|2023-04-05|completed|     1677.58|          20|
| O000008|     C00313|24/10/2021|completed|     1492.92|          25|
| O000009|     C00391|2021-01-28|  pending|      394.15|           5|
| O000010|     C00136|18/11/2023|completed|      118.46|          25|
| O000011|     C00081|2021-01-07|  pending|      354.52|           5|
| O000012|     C0028

In [ ]:
df_orders.printSchema()

In [9]:
returnStruct = StructType ([
    StructField('return_id', StringType(), True),
    StructField('order_id', StringType(), True),
    StructField('return_date', StringType(), True),
    StructField('reason', StringType(), True),
    StructField('refund_amount', FloatType(), True)
])

In [10]:
df_returns = spark.read.format('csv')\
                .schema(returnStruct)\
                .option('inferschema', False)\
                .option('header', True)\
                .load('C:/Users/ADMIN/Desktop/Lux Dev Data Enginering/Assignment_1/data/returns.csv')
df_returns.show()

+---------+--------+-----------+-------------------+-------------+
|return_id|order_id|return_date|             reason|refund_amount|
+---------+--------+-----------+-------------------+-------------+
|  R000001| O001713| 19/04/2024|       changed mind|       349.04|
|  R000002| O001177| 05/08/2022|         wrong item|      1721.96|
|  R000003| O000991| 2021-01-08|   not as described|        27.77|
|  R000004| O000017| 12/01/2023|          defective|       260.67|
|  R000005| O001787| 2021-09-05|          defective|       754.97|
|  R000006| O000732| 20/06/2022|          defective|       240.01|
|  R000007| O001191| 05/06/2021|   not as described|       913.52|
|  R000008| O000666| 2023-01-02|       changed mind|        27.48|
|  R000009| O000147| 2023-05-05|          defective|         71.9|
|  R000010| O000746| 2023-12-03|         wrong item|       165.16|
|  R000011| O001509| 2022-09-05|         wrong item|       492.46|
|  R000012| O001431| 22/12/2023|damaged in shipping|      1730

In [ ]:
df_returns.printSchema()

Task 02: Clean in the exact sequence the spec says deduplicate-> normalize dates -> lowercase tier -> drop nulls -> flag negatives. Handling the mixed date formats(year-month-day vs day-month-year). use F.to/-date with coalesce to try both formats.

In [11]:
# Remove exact duplicate rows using dropDuplicates().
df_customers = df_customers.dropDuplicates()
df_order_items = df_order_items.dropDuplicates()
df_returns = df_returns.dropDuplicates()
df_orders = df_orders.dropDuplicates()

In [12]:
df_customers = df_customers.withColumn(
    'signup_date',
    coalesce(
        try_to_date(col("signup_date"), "yyyy-MM-dd"),
        try_to_date(col("signup_date"), "dd/MM/yyyy")
    )
)

In [ ]:
df_customers.show()

In [13]:
df_orders = df_orders.withColumn(
    'order_date',
    coalesce(
        try_to_date(col("order_date"), "yyyy-MM-dd"),
        try_to_date(col("order_date"), "dd/MM/yyyy")
    )
  )
df_orders.show()

+--------+-----------+----------+---------+------------+------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|
+--------+-----------+----------+---------+------------+------------+
| O000398|     C00066|2022-06-15|  pending|      736.44|          20|
| O000531|     C00355|2023-02-22|completed|       19.92|           5|
| O000664|     C00257|2024-08-07|cancelled|      605.83|          25|
| O000727|     C00055|2023-11-06|completed|      353.01|           0|
| O000916|     C00073|2021-12-09|cancelled|      463.34|          25|
| O001307|     C00350|2024-06-03|  pending|      650.34|           0|
| O001930|     C00107|2022-11-18|cancelled|      569.28|          20|
| O001933|     C00193|2022-10-03|  pending|      899.03|          25|
| O000072|     C00254|2023-09-22|  pending|      668.16|           0|
| O000167|     C00235|2024-11-06|completed|     1781.63|          25|
| O000442|     C00136|2023-11-11|cancelled|      597.83|          25|
| O000521|     C0015

In [14]:
## Standardise customer_tier to lowercase.
df_customers = df_customers.withColumn("customer_tier", lower("customer_tier"))
df_customers.show()

+-----------+-------------------+--------------------+-------+-------------+-----------+
|customer_id|      customer_name|               email|country|customer_tier|signup_date|
+-----------+-------------------+--------------------+-------+-------------+-----------+
|     C00059|      Don Tucker MD| dramsey@example.org|     VU|       bronze| 2019-11-17|
|     C00173|       Eric Sanders|   wking@example.net|     ET|       silver| 2018-05-11|
|     C00174|      Richard Glenn|joshua56@example.com|     CH|         gold| 2020-06-07|
|     C00076| Dr. Elizabeth Ward|alexis82@example.com|     BD|         gold| 2020-05-29|
|     C00147|    Victoria Morris|edward17@example.com|     ZM|       silver| 2022-10-20|
|     C00181|     Kendra Sanders|  john10@example.net|     ER|       bronze| 2018-08-27|
|     C00151|Mrs. Maria Williams| hritter@example.net|     VA|       silver| 2020-12-03|
|     C00093|       Adam Russell|victoriadurham@ex...|     CL|       silver| 2019-05-08|
|     C00092|        

In [15]:
## Drop rows where order_id or customer_id is NULL.
df_customers = df_customers.dropna(subset=["customer_id"])
df_order_items = df_order_items.dropna(subset=["order_id"])
df_returns = df_returns.dropna(subset=["order_id"])
df_orders = df_orders.dropna(subset=["order_id", "customer_id"])
df_customers.show()

+-----------+-------------------+--------------------+-------+-------------+-----------+
|customer_id|      customer_name|               email|country|customer_tier|signup_date|
+-----------+-------------------+--------------------+-------+-------------+-----------+
|     C00059|      Don Tucker MD| dramsey@example.org|     VU|       bronze| 2019-11-17|
|     C00173|       Eric Sanders|   wking@example.net|     ET|       silver| 2018-05-11|
|     C00174|      Richard Glenn|joshua56@example.com|     CH|         gold| 2020-06-07|
|     C00076| Dr. Elizabeth Ward|alexis82@example.com|     BD|         gold| 2020-05-29|
|     C00147|    Victoria Morris|edward17@example.com|     ZM|       silver| 2022-10-20|
|     C00181|     Kendra Sanders|  john10@example.net|     ER|       bronze| 2018-08-27|
|     C00151|Mrs. Maria Williams| hritter@example.net|     VA|       silver| 2020-12-03|
|     C00093|       Adam Russell|victoriadurham@ex...|     CL|       silver| 2019-05-08|
|     C00092|        

In [16]:
## Add a boolean column is_negative_amount to flag (not drop) orders with total_amount < 0.
df_orders = df_orders.withColumn("is_negative_amount", when((col('total_amount') < 0), "True").otherwise("False"))
df_orders.show()


+--------+-----------+----------+---------+------------+------------+------------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|is_negative_amount|
+--------+-----------+----------+---------+------------+------------+------------------+
| O000398|     C00066|2022-06-15|  pending|      736.44|          20|             False|
| O000531|     C00355|2023-02-22|completed|       19.92|           5|             False|
| O000664|     C00257|2024-08-07|cancelled|      605.83|          25|             False|
| O000727|     C00055|2023-11-06|completed|      353.01|           0|             False|
| O000916|     C00073|2021-12-09|cancelled|      463.34|          25|             False|
| O001307|     C00350|2024-06-03|  pending|      650.34|           0|             False|
| O001930|     C00107|2022-11-18|cancelled|      569.28|          20|             False|
| O001933|     C00193|2022-10-03|  pending|      899.03|          25|             False|
| O000072|     C00254

## Task 03 and 04
### Joins + Aggregations

Task 03: Three joins - orders-> customers(inner), orders->order_items(left), then an anti-join to catch orphaned itens. Derive net_amount as a new column

In [17]:
df_order_customer= df_orders.join(df_customers, on=["customer_id"], how="inner")


In [ ]:
df_order_customer.show()

In [18]:
df_orders_order_items = df_orders.join(df_order_items, on=["order_id"], how="left")
df_orders_order_items.show()

+--------+-----------+----------+---------+------------+------------+------------------+--------+-------------------+-------------+--------+----------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|is_negative_amount| item_id|       product_name|     category|quantity|unit_price|
+--------+-----------+----------+---------+------------+------------+------------------+--------+-------------------+-------------+--------+----------+
| O000398|     C00066|2022-06-15|  pending|      736.44|          20|             False|I0001203|      Support Voice|         Toys|       2|    247.28|
| O000398|     C00066|2022-06-15|  pending|      736.44|          20|             False|I0001202|      Good Practice|  Electronics|       7|    154.14|
| O000531|     C00355|2023-02-22|completed|       19.92|           5|             False|I0001616|    Throughout Skin|       Sports|       2|    224.37|
| O000531|     C00355|2023-02-22|completed|       19.92|           5|             False|

In [19]:
df_orphaned_order_items = df_orders.join(df_order_items, on=["order_id"], how="anti")
df_orphaned_order_items.show()

+--------+-----------+----------+------+------------+------------+------------------+
|order_id|customer_id|order_date|status|total_amount|discount_pct|is_negative_amount|
+--------+-----------+----------+------+------------+------------+------------------+
+--------+-----------+----------+------+------------+------------+------------------+



In [20]:
df_orphaned_customers = df_orders.join(df_customers, on=["customer_id"], how="anti")
df_orphaned_customers.show()

+-----------+--------+----------+------+------------+------------+------------------+
|customer_id|order_id|order_date|status|total_amount|discount_pct|is_negative_amount|
+-----------+--------+----------+------+------------+------------+------------------+
+-----------+--------+----------+------+------------+------------+------------------+



In [21]:
# Add a derived column net_amount = total_amount × (1 − discount_pct / 100)
df_order_customer = df_order_customer.withColumn('net_amount', (col("total_amount") * (1 - col("discount_pct")/100)))
df_order_customer.show()

+-----------+--------+----------+---------+------------+------------+------------------+-------------------+--------------------+-------+-------------+-----------+------------------+
|customer_id|order_id|order_date|   status|total_amount|discount_pct|is_negative_amount|      customer_name|               email|country|customer_tier|signup_date|        net_amount|
+-----------+--------+----------+---------+------------+------------+------------------+-------------------+--------------------+-------+-------------+-----------+------------------+
|     C00066| O000398|2022-06-15|  pending|      736.44|          20|             False|  Raymond Jefferson|hoganashlee@examp...|     LY|         gold| 2020-01-25|  589.152001953125|
|     C00355| O000531|2023-02-22|completed|       19.92|           5|             False|      Nicholas Rush|xrosales@example.net|     AU|       silver| 2020-07-04| 18.92400007247925|
|     C00257| O000664|2024-08-07|cancelled|      605.83|          25|             Fal

### Task 4 - Three window function outputs. partitionBy and orderBy for each - rank by country, rolling 7-day count, monthly revenue share per category. 

In [22]:
#  Customers ranked by lifetime net spend within each country.
## columns to use - net_amount, country, customer_id

from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [23]:
windowSpec = Window.partitionBy("customer_name").orderBy(F.desc("net_amount"))



In [24]:
# 1. Aggregate to calculate total lifetime spend per customer per country
df_lifetime = df_order_customer.groupBy("country", "customer_id", "customer_name") \
                .agg(F.sum("net_amount").alias("lifetime_net_spend"))

# 2. Define the Window to rank by spend within each country
# Highest spend gets rank 1
window_spec = Window.partitionBy("country").orderBy(F.col("lifetime_net_spend").desc())

# 3. Apply dense_rank() to handle potential spending ties gracefully
df_ranked = df_lifetime.withColumn("spend_rank", F.dense_rank().over(window_spec))

# Display the final ranking
df_ranked.orderBy("country", "spend_rank").show()

+-------+-----------+-------------------+------------------+----------+
|country|customer_id|      customer_name|lifetime_net_spend|spend_rank|
+-------+-----------+-------------------+------------------+----------+
|     AD|     C00090|       Brandon King|  10061.3364944458|         1|
|     AD|     C00047|      Jason Bentley| 9394.521502685548|         2|
|     AD|     C00300|      Melissa Wiley| 2176.942980957031|         3|
|     AD|     C00358|       Derek Wright| 105.8395034790039|         4|
|     AE|     C00331|        David Bruce| 4833.043542480469|         1|
|     AE|     C00309|      Justin Bishop|  3083.97553024292|         2|
|     AE|     C00312|     Melissa Abbott|2899.2920364379884|         3|
|     AE|     C00178|      William Gould|1725.6214569091799|         4|
|     AE|     C00010|       Thomas Ellis| 592.8674926757812|         5|
|     AG|     C00385|        Debra Ortiz| 3740.955453491211|         1|
|     AL|     C00122|     William Huerta|10240.632481193543|    

In [25]:
# A 7-day rolling order count per customer (based on order_date).

df_oc = df_order_customer.withColumn("date_in_seconds", F.col("order_date").cast("timestamp").cast("long"))

seconds_in_7_days = 6 * 24 * 60 * 60 

window_spec = Window.partitionBy("customer_id") \
                    .orderBy("date_in_seconds") \
                    .rangeBetween(-seconds_in_7_days, Window.currentRow)

df_rolling = df_oc.withColumn(
    "rolling_7day_order_count", 
    F.count("order_id").over(window_spec)
).drop("date_in_seconds")

df_rolling.orderBy("customer_id", "order_date").show()

+-----------+--------+----------+---------+------------+------------+------------------+--------------+--------------------+-------+-------------+-----------+------------------+------------------------+
|customer_id|order_id|order_date|   status|total_amount|discount_pct|is_negative_amount| customer_name|               email|country|customer_tier|signup_date|        net_amount|rolling_7day_order_count|
+-----------+--------+----------+---------+------------+------------+------------------+--------------+--------------------+-------+-------------+-----------+------------------+------------------------+
|     C00001| O000989|2022-05-31|  pending|      798.16|          10|             False| Donald Walker|rhodespatricia@ex...|     UA|         gold| 2018-12-12| 718.3439758300782|                       1|
|     C00001| O001685|2022-06-17| refunded|      848.71|          25|             False| Donald Walker|rhodespatricia@ex...|     UA|         gold| 2018-12-12| 636.5325164794922|           

In [26]:
# Each product category’s share of total revenue per calendar month.
df_prod_cat = df_orders_order_items.groupBy("category", "order_date").agg(F.sum("total_amount").alias("total_revenue"))

# 3. Define a Window partitioned ONLY by month to sum up all categories together
monthly_window = Window.partitionBy("order_date")

# 4. Calculate the total monthly revenue and divide the category share
df_share = df_prod_cat \
    .withColumn("total_monthly_revenue", F.sum("total_revenue").over(monthly_window)) \
    .withColumn("revenue_share_pct", F.round((F.col("total_revenue") / F.col("total_monthly_revenue")) * 100, 2))

# Display the result sorted chronologically
df_share.orderBy("order_date", F.col("revenue_share_pct").desc()).show()

+-------------+----------+------------------+---------------------+-----------------+
|     category|order_date|     total_revenue|total_monthly_revenue|revenue_share_pct|
+-------------+----------+------------------+---------------------+-----------------+
|        Books|2021-01-01|1058.4300231933594|   3899.7901306152344|            27.14|
|       Sports|2021-01-01| 710.3400268554688|   3899.7901306152344|            18.21|
|     Clothing|2021-01-01| 710.3400268554688|   3899.7901306152344|            18.21|
|Home & Garden|2021-01-01| 710.3400268554688|   3899.7901306152344|            18.21|
|         Toys|2021-01-01| 710.3400268554688|   3899.7901306152344|            18.21|
|     Clothing|2021-01-02| 1954.449951171875|    1954.449951171875|            100.0|
|         Toys|2021-01-04| 2095.340087890625|   5238.3502197265625|             40.0|
|       Sports|2021-01-04|1047.6700439453125|   5238.3502197265625|             20.0|
|        Books|2021-01-04|1047.6700439453125|   5238.3

In [27]:
## Join returns to your enriched orders DataFrame
df_order_customer = df_order_customer.join(df_ranked, on=["customer_id"], how="inner")
df_order_customer.show(10)

+-----------+--------+----------+---------+------------+------------+------------------+-------------+--------------------+-------+-------------+-----------+------------------+-------+-------------+------------------+----------+
|customer_id|order_id|order_date|   status|total_amount|discount_pct|is_negative_amount|customer_name|               email|country|customer_tier|signup_date|        net_amount|country|customer_name|lifetime_net_spend|spend_rank|
+-----------+--------+----------+---------+------------+------------+------------------+-------------+--------------------+-------+-------------+-----------+------------------+-------+-------------+------------------+----------+
|     C00090| O000940|2021-06-02| refunded|       595.7|           0|             False| Brandon King|cordovarichard@ex...|     AD|      silver | 2022-09-06| 595.7000122070312|     AD| Brandon King|  10061.3364944458|         1|
|     C00090| O001922|2022-04-27|cancelled|     1807.26|           5|             Fa

In [28]:
df_orders_order_items.show(5)

+--------+-----------+----------+---------+------------+------------+------------------+--------+-------------------+-------------+--------+----------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|is_negative_amount| item_id|       product_name|     category|quantity|unit_price|
+--------+-----------+----------+---------+------------+------------+------------------+--------+-------------------+-------------+--------+----------+
| O000398|     C00066|2022-06-15|  pending|      736.44|          20|             False|I0001203|      Support Voice|         Toys|       2|    247.28|
| O000398|     C00066|2022-06-15|  pending|      736.44|          20|             False|I0001202|      Good Practice|  Electronics|       7|    154.14|
| O000531|     C00355|2023-02-22|completed|       19.92|           5|             False|I0001616|    Throughout Skin|       Sports|       2|    224.37|
| O000531|     C00355|2023-02-22|completed|       19.92|           5|             False|

In [29]:
## Compute return rates using groupBy aggregations
df_return_rates = df_orders_order_items.groupBy("category").agg(
    # 1. Total orders in this category
    F.count("order_id").alias("total_orders"),

    # 2. Returned orders — no .otherwise() so non-matches become NULL and count skips them
    F.count(
        F.when(F.col("status") == "refunded", F.col("order_id"))
    ).alias("total_returned")

).withColumn(
    # 3. Return rate percentage
    "return_rate_pct",
    F.round((F.col("total_returned") / F.col("total_orders")) * 100, 2)
)

df_return_rates.orderBy(F.col("return_rate_pct").desc()).show()

+-------------+------------+--------------+---------------+
|     category|total_orders|total_returned|return_rate_pct|
+-------------+------------+--------------+---------------+
|Home & Garden|         880|           223|          25.34|
|     Clothing|         820|           207|          25.24|
|       Sports|         761|           192|          25.23|
|        Books|         778|           194|          24.94|
|       Beauty|         800|           191|          23.88|
|  Electronics|         836|           197|          23.56|
|         Toys|         887|           200|          22.55|
+-------------+------------+--------------+---------------+



In [ ]:
## Flag where refund_amount > net_amount


In [ ]:
## Get top 10 refund customers


In [31]:
df_order_customer = df_order_customer.drop("country", "customer_name")  # drop one of the duplicates

In [32]:
df_order_customer.write \
    .mode('overwrite') \
    .parquet('C:/Users/ADMIN/Desktop/Assignment_1/')
    

Py4JJavaError: An error occurred while calling o306.parquet.
: java.util.concurrent.ExecutionException: Boxed Exception
	at scala.concurrent.impl.Promise$.scala$concurrent$impl$Promise$$resolve(Promise.scala:99)
	at scala.concurrent.impl.Promise$DefaultPromise.tryComplete(Promise.scala:288)
	at scala.concurrent.Promise.complete(Promise.scala:57)
	at scala.concurrent.Promise.complete$(Promise.scala:56)
	at scala.concurrent.impl.Promise$DefaultPromise.complete(Promise.scala:104)
	at scala.concurrent.Promise.failure(Promise.scala:109)
	at scala.concurrent.Promise.failure$(Promise.scala:109)
	at scala.concurrent.impl.Promise$DefaultPromise.failure(Promise.scala:104)
	at org.apache.spark.sql.execution.adaptive.ResultQueryStageExec.$anonfun$doMaterialize$2(QueryStageExec.scala:336)
	at java.base/java.util.concurrent.CompletableFuture.uniWhenComplete(CompletableFuture.java:863)
	at java.base/java.util.concurrent.CompletableFuture$UniWhenComplete.tryFire(CompletableFuture.java:841)

In [33]:
# Option 1: Using file:// URI scheme for local file system
df_order_customer.write \
    .mode('overwrite') \
    .parquet('file:///C:/Users/ADMIN/Desktop/Assignment_1/')

# Option 2: Using forward slashes (recommended for cross-platform compatibility)
df_order_customer.write \
    .mode('overwrite') \
    .parquet('C:/Users/ADMIN/Desktop/Assignment_1/')

# Option 3: Using raw string to handle backslashes properly
df_order_customer.write \
    .mode('overwrite') \
    .parquet(r'C:\Users\ADMIN\Desktop\Assignment_1')

# Option 4: Create directory first and use relative path
import os
output_path = 'C:/Users/ADMIN/Desktop/Assignment_1/'
os.makedirs(output_path, exist_ok=True)  # Create directory if it doesn't exist
df_order_customer.write \
    .mode('overwrite') \
    .parquet(output_path)

Py4JJavaError: An error occurred while calling o312.parquet.
: java.util.concurrent.ExecutionException: Boxed Exception
	at scala.concurrent.impl.Promise$.scala$concurrent$impl$Promise$$resolve(Promise.scala:99)
	at scala.concurrent.impl.Promise$DefaultPromise.tryComplete(Promise.scala:288)
	at scala.concurrent.Promise.complete(Promise.scala:57)
	at scala.concurrent.Promise.complete$(Promise.scala:56)
	at scala.concurrent.impl.Promise$DefaultPromise.complete(Promise.scala:104)
	at scala.concurrent.Promise.failure(Promise.scala:109)
	at scala.concurrent.Promise.failure$(Promise.scala:109)
	at scala.concurrent.impl.Promise$DefaultPromise.failure(Promise.scala:104)
	at org.apache.spark.sql.execution.adaptive.ResultQueryStageExec.$anonfun$doMaterialize$2(QueryStageExec.scala:336)
	at java.base/java.util.concurrent.CompletableFuture.uniWhenComplete(CompletableFuture.java:863)
	at java.base/java.util.concurrent.CompletableFuture$UniWhenComplete.tryFire(CompletableFuture.java:841)

In [ ]:
## writing summary tables to csv
df_return_rates.write('csv')\
    .mode('overwrite')\
    .option('path', 'C:/Users/ADMIN/Desktop/Assignment_1/output/return_rates.csv')\
    .save()

In [ ]:
## writing summary tables to csv
df_share.write('csv')\
    .mode('overwrite')\
    .option('path', 'C:\Users\ADMIN\Desktop\Lux Dev Data Enginering\Assignment_1\pc_share.csv')\
    .save()


This would prevent the TypeError by first checking if `val[0]` is not None before attempting to search for the '/' character within it.


Would you like me to provide the corrected code?

In [ ]:
df_customers.show()

In [ ]:
df_customers.printSchema()

In [ ]:
orderItemStruct = StructType([
    StructField('item_id', StringType(), True),
    StructField('order_id', StringType(), True),
    StructField('product_name', StringType(), True),
    StructField('category', StringType(), True),
    StructField('quantity', IntegerType(), True),
    StructField('unit_price', FloatType(), True),
])

In [ ]:
df_order_items = spark.read.format('csv')\
                .schema(orderItemStruct)\
                .option('inferschema', False)\
                .option('header', True)\
                .load('C:/Users/ADMIN/Desktop/Lux Dev Data Enginering/Assignment_1/data/order_items.csv')

In [ ]:
df_order_items.show()

In [ ]:
df_order_items.printSchema()

In [ ]:
orderStruct = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("status", StringType(), True),
    StructField("total_amount", FloatType(), True),
    StructField("discount_pct",IntegerType(), True)
    ])

In [ ]:
df_orders = spark.read.format('csv')\
                .schema(orderStruct)\
                .option('inferschema', False)\
                .option('header', True)\
                .load('C:/Users/ADMIN/Desktop/Lux Dev Data Enginering/Assignment_1/data/orders.csv')
df_orders.show()

In [ ]:
df_orders.printSchema()

In [ ]:
returnStruct = StructType ([
    StructField('return_id', StringType(), True),
    StructField('order_id', StringType(), True),
    StructField('return_date', StringType(), True),
    StructField('reason', StringType(), True),
    StructField('refund_amount', FloatType(), True)
])

In [ ]:
df_returns = spark.read.format('csv')\
                .schema(returnStruct)\
                .option('inferschema', False)\
                .option('header', True)\
                .load('C:/Users/ADMIN/Desktop/Lux Dev Data Enginering/Assignment_1/data/returns.csv')
df_returns.show()

In [ ]:
df_returns.printSchema()

note: while designing the structtypes for date columns, since the dates have a different order, they are interpreted as null thus string type was the option

Task 02: Clean in the exact sequence the spec says deduplicate-> normalize dates -> lowercase tier -> drop nulls -> flag negatives. Handling the mixed date formats(year-month-day vs day-month-year). use F.to/-date with coalesce to try both formats.

In [ ]:
df_customers = df_customers.dropDuplicates().show()

In [ ]:
df_order_items = df_order_items.dropDuplicates().show()

In [ ]:
df_returns = df_returns.dropDuplicates().show()

In [ ]:
df_orders = df_orders.dropDuplicates().show()

In [ ]:
df_orders.withColumn('formatted_order_date', date_format('order_date', 'yyyy-MM-dd')).show()